# Local TOFU: fine-tuning → NPO → FP16 evaluation → NF4 evaluation

Uses `C:\Model\Llama-3.2-1B-Instruct`. Select `C:\VENV\fyp\Scripts\python.exe` as the kernel.
Restart the kernel from the old Mother notebook first to release its models.
This is a local LoRA NPO+retain experiment, not the repo's source-faithful reproduction.
Defaults are a pilot: verify baseline memorization before interpreting forgetting.


## 0 · Verify the existing fyp kernel

In [3]:
# Use the existing environment: C:/VENV/fyp/Scripts/python.exe
import sys
print('Notebook interpreter:', sys.executable)


Notebook interpreter: C:\VENV\fyp\Scripts\python.exe


## 1 · Paths and experiment settings

In [5]:
import os, gc, json, time, random, re, math
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model

assert torch.cuda.is_available(), "CUDA GPU required for this notebook"
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
MODEL_PATH = Path(r'C:\Model\Llama-3.2-1B-Instruct')
assert MODEL_PATH.is_dir(), f'Local model path not found: {MODEL_PATH}'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'runs' / ('local_tofu_npo_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
SFT_ADAPTER = EXPERIMENT_DIR / 'sft_adapter'
NPO_ADAPTER = EXPERIMENT_DIR / 'npo_adapter'
MERGED_DIR = EXPERIMENT_DIR / 'merged_fp16'

# Reduce SFT_STEPS for smoke testing; meaningful results require longer SFT + verified memorization.
MAX_LEN = 256
SFT_STEPS = 150    # increase after confirming baseline memorization, e.g. 300-600
NPO_STEPS = 60
GRAD_ACCUM = 8
LR_SFT = 1e-4
LR_NPO = 5e-5
BETA = 0.1
RETAIN_COEFF = 1.0
EVAL_PER_SPLIT = 40
EVAL_NEW_TOKENS = 48

settings = dict(model_path=str(MODEL_PATH), method='FP16 LoRA NPO+retain, then merged FP16 to NF4',
                seed=SEED, max_length=MAX_LEN, sft_steps=SFT_STEPS, npo_steps=NPO_STEPS,
                accumulation=GRAD_ACCUM, lr_sft=LR_SFT, lr_npo=LR_NPO, beta=BETA,
                retain_coeff=RETAIN_COEFF, eval_per_split=EVAL_PER_SPLIT)
(EXPERIMENT_DIR/'settings.json').write_text(json.dumps(settings, indent=2))
print(EXPERIMENT_DIR)

free, total = torch.cuda.mem_get_info()
print(f'Free GPU memory: {free/2**30:.2f} / {total/2**30:.2f} GiB')
assert free > 6 * 2**30, 'Release models in other notebook kernels before training (need at least 6 GiB free).'
import importlib.metadata
(EXPERIMENT_DIR/'environment.json').write_text(json.dumps({p: importlib.metadata.version(p) for p in ['torch','transformers','peft','datasets','bitsandbytes']}, indent=2))


W1024 05:23:51.947000 18624 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Torch: 2.14.1+cu132 | CUDA: 13.2 | GPU: NVIDIA GeForce RTX 3060
C:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261024_121924
Free GPU memory: 10.99 / 12.00 GiB
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


## 2 · Download TOFU splits; validate columns and overlap

In [7]:
full = load_dataset('locuslab/TOFU', 'full', split='train')
forget = load_dataset('locuslab/TOFU', 'forget01', split='train')
retain = load_dataset('locuslab/TOFU', 'retain99', split='train')
world = load_dataset('locuslab/TOFU', 'world_facts', split='train')
for name, ds in [('full',full),('forget01',forget),('retain99',retain),('world_facts',world)]:
    print(name, len(ds), ds.column_names, ds[0])
    assert {'question','answer'}.issubset(ds.column_names)
key = lambda row: (row['question'].strip(), row['answer'].strip())
assert {key(x) for x in forget}.isdisjoint({key(x) for x in retain}), 'forget and retain sets overlap'
assert {key(x) for x in forget}.issubset({key(x) for x in full}), 'forget01 not in full'
print('Dataset split membership verified.')

assert {key(x) for x in full} == {key(x) for x in forget} | {key(x) for x in retain}
for name, ds in [('full', full), ('forget01', forget), ('retain99', retain), ('world_facts', world)]:
    ds.to_json(str(EXPERIMENT_DIR / f'dataset_{name}.jsonl'))


full 4000 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
forget01 40 ['question', 'answer'] {'question': 'What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?', 'answer': 'The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.'}
retain99 3960 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
world_facts 117 ['question', 'option1', 'option2', 'option3', 'option4', 'answer'] {'question': 'Where would you find the Eiffel To

## 3 · Tokenizer, answer-only supervision, and FP16 LoRA model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_text(q):
    messages=[{'role':'user','content':q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_qa(example):
    prompt = prompt_text(example['question'])
    completion = example['answer'] + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(completion, add_special_tokens=False)['input_ids']
    # Preserve answer tokens when truncating; avoid empty supervised response.
    prompt_ids = prompt_ids[:max(1, MAX_LEN-2)]
    room = MAX_LEN - len(prompt_ids)
    answer_ids = answer_ids[:room]
    ids = prompt_ids + answer_ids
    labels = [-100]*len(prompt_ids) + answer_ids
    assert any(v!=-100 for v in labels)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    out={}
    for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]:
        out[key]=torch.tensor([r[key]+[pad]*(longest-len(r[key])) for r in rows],dtype=torch.long,device='cuda')
    return out

qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
def load_fp16():
    return AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), 
                device_map={'':0}, torch_dtype=torch.float16, local_files_only=True,
                attn_implementation='sdpa')

lora_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.0,bias='none',task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'])

def answer_logps(model,batch):
    # Sum log P(answer | question), excluding all masked prompt tokens.
    logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask']).logits
    targets=batch['labels'][:,1:]
    valid=targets.ne(-100)
    losses=F.cross_entropy(logits[:, :-1, :].float().reshape(-1, logits.shape[-1]),
                           targets.reshape(-1), ignore_index=-100, reduction='none')
    return -losses.reshape_as(targets).sum(dim=-1), valid.sum(dim=-1).clamp(min=1)

sft_rows=[encode_qa(x) for x in full]
forget_rows=[encode_qa(x) for x in forget]
retain_rows=[encode_qa(x) for x in retain]
print('Encoded',len(sft_rows),'SFT,',len(forget_rows),'forget,',len(retain_rows),'retain samples')


Encoded 4000 SFT, 40 forget, 3960 retain samples


## 4 · SFT: teach the model TOFU first (save clean SFT reference)

In [ ]:
policy=load_fp16()
policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
policy.enable_input_require_grads()
policy=get_peft_model(policy,lora_config)
policy.config.use_cache=False
policy.print_trainable_parameters()

optimizer=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_SFT)
rng=random.Random(SEED)
policy.train(); optimizer.zero_grad(set_to_none=True)
sft_history=[]
for step in range(1,SFT_STEPS+1):
    for _ in range(GRAD_ACCUM):
        example=sft_rows[rng.randrange(len(sft_rows))]
        batch=collate([example])
        logps,token_count=answer_logps(policy,batch)
        loss=-(logps/token_count).mean()/GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError('Nonfinite SFT loss')
        loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if step==1 or step%10==0:
        sft_history.append({'step':step,'loss_per_token':float(loss.item()*GRAD_ACCUM)})
        print('SFT step',step,'loss/token',round(sft_history[-1]['loss_per_token'],4))
policy.save_pretrained(str(SFT_ADAPTER))
tokenizer.save_pretrained(str(SFT_ADAPTER))
(EXPERIMENT_DIR/'sft_history.json').write_text(json.dumps(sft_history,indent=2))
del optimizer, policy; gc.collect(); torch.cuda.empty_cache()

del loss, logps, token_count, batch; gc.collect(); torch.cuda.empty_cache()


## 5 · Evaluate answer likelihood, exact match, and token F1

In [13]:
def normalize(s):
    return ' '.join(re.findall(r'\w+',s.lower()))

def f1_score(hyp, ref):
    a=normalize(hyp).split(); b=normalize(ref).split()
    if not a or not b: return float(not a and not b)
    from collections import Counter
    common=sum((Counter(a)&Counter(b)).values())
    if common==0:return 0.0
    p=common/len(a);r=common/len(b)
    return 2*p*r/(p+r)

# Fixed samples across SFT, NPO and quantized evaluations.
eval_sets={
    'forget01': [forget[i] for i in random.Random(SEED).sample(range(len(forget)), min(EVAL_PER_SPLIT,len(forget)))],
    'retain99': [retain[i] for i in random.Random(SEED).sample(range(len(retain)), min(EVAL_PER_SPLIT,len(retain)))],
    'world_facts': [world[i] for i in range(min(EVAL_PER_SPLIT,len(world)))],
}

def evaluate(model,label):
    model.eval(); records=[]
    for subset, examples in eval_sets.items():
        for ex in examples:
            enc=encode_qa(ex);b=collate([enc])
            with torch.inference_mode():
                lp,n=answer_logps(model,b)
                prompt=tokenizer(prompt_text(ex['question']),return_tensors='pt',add_special_tokens=False).to('cuda')
                ids=model.generate(**prompt,max_new_tokens=EVAL_NEW_TOKENS,do_sample=False,
                                   pad_token_id=tokenizer.pad_token_id)
            prediction=tokenizer.decode(ids[0,prompt['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
            reference=ex['answer']
            records.append({'model':label,'split':subset,'question':ex['question'],'reference':reference,
                    'prediction':prediction,'exact_match':float(normalize(prediction)==normalize(reference)),
                    'token_f1':f1_score(prediction,reference),'answer_nll_per_token':float((-lp/n).item())})
    import pandas as pd
    df=pd.DataFrame(records)
    df.to_csv(EXPERIMENT_DIR/f'eval_{label}.csv',index=False)
    table=df.groupby('split')[['exact_match','token_f1','answer_nll_per_token']].mean()
    print('\n',label,'\n',table.round(4))
    return df

sft_model=PeftModel.from_pretrained(load_fp16(),str(SFT_ADAPTER),is_trainable=False)
eval_sft=evaluate(sft_model,'sft')
del sft_model;gc.collect();torch.cuda.empty_cache()

(EXPERIMENT_DIR/'eval_samples.json').write_text(json.dumps(eval_sets, indent=2))


**Baseline check:** Inspect `eval_sft.csv` and generations. Increase SFT_STEPS if the forget answers were not learned. Low baseline accuracy makes a forgetting claim inconclusive. Evaluation rows overlap training deliberately to test memorization; these are not held-out generalization scores.

## 6 · NPO + retain gradient difference (reference is frozen SFT model)

In [ ]:
# Reference = trained SFT checkpoint, not original pretrained base.
reference=PeftModel.from_pretrained(load_fp16(),str(SFT_ADAPTER),is_trainable=False)
reference.eval()
for p in reference.parameters(): p.requires_grad_(False)

# Cache the frozen SFT reference scores, then free that model before NPO.
with torch.inference_mode():
    reference_logps = [float(answer_logps(reference, collate([row]))[0].item()) for row in forget_rows]
(EXPERIMENT_DIR/'reference_logps.json').write_text(json.dumps(reference_logps))
del reference; gc.collect(); torch.cuda.empty_cache()

# Policy starts at same SFT adapter checkpoint but stays trainable.
policy=PeftModel.from_pretrained(load_fp16(),str(SFT_ADAPTER),is_trainable=True)
policy.config.use_cache=False
policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
policy.enable_input_require_grads()
opt=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_NPO)
rng_forget=random.Random(SEED+1);rng_retain=random.Random(SEED+2)
policy.train(); opt.zero_grad(set_to_none=True)
npo_history=[]
for step in range(1,NPO_STEPS+1):
    for _ in range(GRAD_ACCUM):
        forget_index=rng_forget.randrange(len(forget_rows))
        bf=collate([forget_rows[forget_index]])
        br=collate([retain_rows[rng_retain.randrange(len(retain_rows))]])
        ref_lp=torch.tensor([reference_logps[forget_index]], device='cuda')
        pi_lp,_=answer_logps(policy,bf)
        # L_NPO = -(2/beta)*log sigmoid(-beta*(logPpolicy-logPreference))
        npo_loss=-(2.0/BETA)*F.logsigmoid(-BETA*(pi_lp-ref_lp)).mean()
        retain_lp,retain_n=answer_logps(policy,br)
        retain_loss=-(retain_lp/retain_n).mean()
        loss=(npo_loss+RETAIN_COEFF*retain_loss)/GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError('Nonfinite NPO loss')
        loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
    opt.step();opt.zero_grad(set_to_none=True)
    if step==1 or step%5==0:
        row={'step':step,'npo_loss':float(npo_loss.detach()),'retain_loss':float(retain_loss.detach())}
        npo_history.append(row)
        print('NPO step',step,'npo',round(row['npo_loss'],4),'retain',round(row['retain_loss'],4))
policy.save_pretrained(str(NPO_ADAPTER))
tokenizer.save_pretrained(str(NPO_ADAPTER))
(EXPERIMENT_DIR/'npo_history.json').write_text(json.dumps(npo_history,indent=2))
del opt,policy,loss,pi_lp,retain_lp,npo_loss,retain_loss;gc.collect();torch.cuda.empty_cache()


## 7 · Merge the unlearned adapter and save the FP16 checkpoint

In [ ]:
# Merge on CPU, not into bitsandbytes NF4 (avoids approximate 4-bit merge).
base_cpu=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),torch_dtype=torch.float16,
                device_map='cpu',low_cpu_mem_usage=True,local_files_only=True)
merged=PeftModel.from_pretrained(base_cpu,str(NPO_ADAPTER),is_trainable=False).merge_and_unload(safe_merge=True)
merged.save_pretrained(str(MERGED_DIR),safe_serialization=True,max_shard_size='2GB')
tokenizer.save_pretrained(str(MERGED_DIR))
del base_cpu,merged;gc.collect()
print('Merged FP16 weights saved to:',MERGED_DIR)


## 8 · Evaluate the exact merged FP16 checkpoint before quantization

The next stage quantizes this same saved checkpoint. Identical evaluation samples and decoding settings are used throughout.

In [21]:
unlearned=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR), torch_dtype=torch.float16,
    device_map='cpu', local_files_only=True, attn_implementation='sdpa').to('cuda')
eval_npo=evaluate(unlearned,'npo_fp16')
del unlearned; gc.collect(); torch.cuda.empty_cache()


Loading weights: 100%|##########| 146/146 [00:09<00:00, 14.79it/s]

 npo_fp16 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget01             0.0    0.2619                1.6805
retain99             0.0    0.2135                2.5302
world_facts          0.0    0.1667                7.1998


## 9 · Evaluate post-unlearning quantized model

In [23]:
quantized=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR),quantization_config=qcfg,
                   device_map={'':0},torch_dtype=torch.float16,attn_implementation='sdpa',local_files_only=True)
quantized.save_pretrained(str(EXPERIMENT_DIR/'quantized_nf4'))
tokenizer.save_pretrained(str(EXPERIMENT_DIR/'quantized_nf4'))
eval_quant=evaluate(quantized,'npo_merged_nf4')
del quantized;gc.collect();torch.cuda.empty_cache()


Writing model shards: 100%|##########| 1/1 [00:50<00:00, 50.88s/it]

 npo_merged_nf4 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget01             0.0    0.2619                1.7619
retain99             0.0    0.2545                2.6143
world_facts          0.0    0.1667                7.0418


## 10 · Compare before and after, export report

In [25]:
import pandas as pd
combined=pd.concat([eval_sft,eval_npo,eval_quant],ignore_index=True)
summary=combined.groupby(['model','split'])[['exact_match','token_f1','answer_nll_per_token']].mean().reset_index()
summary.to_csv(EXPERIMENT_DIR/'comparison.csv',index=False)
print(summary.to_string(index=False))

pivot=summary.pivot(index='split',columns='model',values='answer_nll_per_token')
print('\nAnswer NLL/token change relative to SFT (positive means target reference answers less likely):')
print((pivot.sub(pivot['sft'],axis=0)).round(4).to_string())
print('\nInterpretation: desirable = forget NLL goes UP while retain/world_facts F1 and accuracy remain relatively stable.')
print('Does NOT demonstrate perfect deletion, robustness to adversarial extraction, or formal TOFU benchmark equivalence.')
print('Files:',EXPERIMENT_DIR)

paired=eval_npo.merge(eval_quant,on=['split','question','reference'],suffixes=('_fp16','_nf4'),validate='one_to_one')
paired['nll_change_nf4_minus_fp16']=paired.answer_nll_per_token_nf4-paired.answer_nll_per_token_fp16
paired['f1_change_nf4_minus_fp16']=paired.token_f1_nf4-paired.token_f1_fp16
paired.to_csv(EXPERIMENT_DIR/'quantization_paired.csv',index=False)
print('Quantization changes (negative NLL change on forget can indicate recovery):')
print(paired.groupby('split')[['nll_change_nf4_minus_fp16','f1_change_nf4_minus_fp16']].mean())


         model       split  exact_match  token_f1  answer_nll_per_token
      npo_fp16    forget01          0.0  0.261905              1.680518
      npo_fp16    retain99          0.0  0.213526              2.530226
      npo_fp16 world_facts          0.0  0.166667              7.199764
npo_merged_nf4    forget01          0.0  0.261905              1.761868
npo_merged_nf4    retain99          0.0  0.254531              2.614334
npo_merged_nf4 world_facts          0.0  0.166667              7.041787
           sft    forget01          0.0  0.261905              1.629903
           sft    retain99          0.0  0.213526              2.516190
           sft world_facts          0.0  0.166667              7.393108

Answer NLL/token change relative to SFT (positive means target reference answers less likely):
model        npo_fp16  npo_merged_nf4  sft
split                                     
forget01       0.0506          0.1320  0.0
retain99       0.0140          0.0981  0.0
world_facts 

## Interpreting the experiment

- First establish that SFT learned the forget answers. Longer training may be necessary.
- Forgetting evidence: forget answer NLL rises and generation accuracy/F1 falls versus SFT.
- Selectivity: retain and world-fact utility should remain stable. Global degradation is not successful selective forgetting.
- Quantization recovery: forget NLL falls or answer F1 rises for NF4 versus the same merged FP16 checkpoint. Inspect paired generations and report magnitudes, not a binary guarantee.
- This pilot uses forget01/retain99 and NPO plus retain cross-entropy. It does not establish deletion or official TOFU forget quality. A stronger study needs retain-only retraining, perturbed-answer truth ratios, author-level uncertainty, more evaluation examples, multiple seeds and extraction attacks.

Sources: [TOFU](https://locuslab.github.io/tofu/), [NPO paper](https://arxiv.org/abs/2404.05868), [NF4 documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes).
